# Solrへの登録のシンプルなサンプル

Solrをベクトルデータベースとして使用するサンプル。

※Solrのバージョン：9.8.1

Solr 9.0以降でベクトル検索機能が追加され、dense_vector型を使用してベクトル検索が可能になりました。

注意：
solr-vector-ex01-00-prepare.ipynb
を実行し、コアを作成・フィールドを追加したうえで実行してください。

## 必要パッケージのインポート

In [ ]:
import pysolr
import requests
from sentence_transformers import SentenceTransformer
import time
import json

## 設定

In [ ]:
from solr_conf import *

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer(MODEL_NAME)

### スキーマ設定（フィールド追加）

In [ ]:
# まず、ベクトルフィールドタイプを追加
vector_field_type = {
    "add-field-type": {
        "name": "knn_vector",
        "class": "solr.DenseVectorField",
        "vectorDimension": MODEL_DIM,
        "similarityFunction": "cosine",
        "knnAlgorithm": "hnsw"
    }
}

type_response = requests.post(
    f'{SOLR_URL}/{CORE_NAME}/schema',
    json=vector_field_type,
    headers={'Content-Type': 'application/json'}
)

if type_response.status_code == 200:
    try:
        print(f"Vector field type addition response: {type_response.json()}")
    except:
        print(f"Vector field type addition status: {type_response.status_code}")
        print(f"Response text: {type_response.text}")
else:
    print(f"Error adding vector field type: {type_response.status_code}")
    print(f"Response: {type_response.text}")

# テキストフィールドの追加
text_field = {
    "add-field": {
        "name": "text",
        "type": "text_general",
        "stored": True,
        "indexed": True
    }
}

text_response = requests.post(
    f'{SOLR_URL}/{CORE_NAME}/schema',
    json=text_field,
    headers={'Content-Type': 'application/json'}
)

if text_response.status_code == 200:
    try:
        print(f"Text field addition response: {text_response.json()}")
    except:
        print(f"Text field addition status: {text_response.status_code}")
        print(f"Response text: {text_response.text}")
else:
    print(f"Error adding text field: {text_response.status_code}")
    print(f"Response: {text_response.text}")

In [ ]:
# ベクトルフィールドの追加（DenseVectorField型）
vector_field = {
    "add-field": {
        "name": "vector",
        "type": "knn_vector",
        "stored": True,
        "indexed": True
    }
}

vector_response = requests.post(
    f'{SOLR_URL}/{CORE_NAME}/schema',
    json=vector_field,
    headers={'Content-Type': 'application/json'}
)

if vector_response.status_code == 200:
    try:
        print(f"Vector field addition response: {vector_response.json()}")
    except:
        print(f"Vector field addition status: {vector_response.status_code}")
        print(f"Response text: {vector_response.text}")
else:
    print(f"Error adding vector field: {vector_response.status_code}")
    print(f"Response: {vector_response.text}")

## Solrクライアント接続

In [ ]:
solr = pysolr.Solr(f'{SOLR_URL}/{CORE_NAME}', always_commit=True)

## ドキュメントをインデックス

In [ ]:
# --- 登録するテキストデータ ---
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

In [ ]:
documents = []
for i, text in enumerate(texts):
    vector = model.encode(text)
    doc = {
        'id': str(i),
        'text': text,
        'vector': vector.tolist()
    }
    documents.append(doc)
    print(f"Prepared: {text}")

# 一括登録
solr.add(documents)
print(f"Indexed {len(documents)} documents to Solr core '{CORE_NAME}'")

## データ登録確認

In [ ]:
# 全ドキュメント取得
results = solr.search('*:*')
print(f"Total documents: {results.hits}")
for doc in results:
    print(f"ID: {doc['id']}, Text: {doc['text']}")